In [ ]:
import * as tslab from "tslab";
import { readFileSync } from "fs";

const css = readFileSync("../style.css", "utf-8");
tslab.display.html(`<style>${css}</style>`);

In [ ]:
type Variable = string;
type Literal = Variable | ['¬', Variable];
type Clause = Set<Literal>;

# The <a href="https://en.wikipedia.org/wiki/DPLL_algorithm">Davis-Putnam Algorithm</a> with the Jeroslow-Wang Heuristic

This notebook implements the algorithm of Davis and Putnam.  Further details about this algorithm are provided in the lecture notes.

The function `complement(l)` computes the complement of a literal `l`.
If $p$ is a propositional variable, we have the following: 
* $\texttt{complement}(p) = \neg p$,
* $\texttt{complement}(\neg p) = p$.

As we are working with clauses that result form transforming given formulas into *conjunctive normal form* and these clauses do not contain $\top$ or $\bot$, we don't have to bother with $\top$ or $\bot$ in this function.

In [ ]:
function complement(l: Literal): Literal | null {
  // Compute the complement of the literal l.
  if (Array.isArray(l) && l[0] === '¬') return l[1];
  if (typeof l === 'string') return ['¬', l];
  return null;
}

The function `extractVariable(l)` extracts the variable from the literal `l`.
If $p$ is a propositional variable, we have the following: 
* $\texttt{extractVariable}(p) = p$,
* $\texttt{extractVariable}(\neg p) = p$.

In [ ]:
function extractVariable(l: Literal): Variable | null {
  // Extract the variable from the literal l.
  if (Array.isArray(l) && l[0] === '¬') return l[1];
  if (typeof l === 'string') return l;
  return null;
}

The function `arb(S)` returns an arbitrary element from the set `S`.

In [ ]:
function arb<T>(S: Set<T> | ReadonlySet<T>): T | null {
  // Return some member from the set S.
  for (const x of S) {
    return x;
  }
  return null;
}

In TypeScript, when using sets, elements are compared by **reference** (memory address), not by their **content**. 

However, in logical problems like SAT (Boolean satisfiability), we want to treat literals and clauses as **equal if their content matches**, even if they are different objects in memory.

For example, the literal `'p'` and another `'p'` stored at different places should be considered equal. Similarly, two clauses containing the same literals but stored as different sets should be considered equal.

The provided functions solve these problems:

- `literalKey` creates a unique string key representing a literal's content (negated or not), so we can compare literals by their string keys.

- `normalizeClause` removes duplicate literals inside a clause based on their keys so that each literal appears only once in a clause.

- `equalClauses` compares two clauses by checking if they contain the same literals based on keys, ignoring object references.

- `clauseHasLiteral` checks if a clause contains a literal by comparing their keys instead of references.

Using these functions ensures correct semantic equality and set membership checks for clauses and literals, which is necessary to correctly implement SAT solving algorithms in TypeScript.

In [ ]:
// Generates a unique string key for a literal (positive or negated)
function literalKey(lit: Literal): string {
  if (typeof lit === 'string') return lit;
  return `¬${lit[1]}`;
}

// Normalize a clause by removing duplicate literals based on content keys
function normalizeClause(clause: Clause): Clause {
  const map = new Map<string, Literal>();
  for (const lit of clause) {
    map.set(literalKey(lit), lit);
  }
  return new Set(map.values());
}

// Compare two clauses for semantic equality (same literals by content)
function equalClauses(c1: Clause, c2: Clause): boolean {
  if (c1.size !== c2.size) return false;
  for (const lit of c1) {
    if (![...c2].some(l => literalKey(l) === literalKey(lit))) return false;
  }
  return true;
}

// Check if a clause contains a literal based on content key, not object reference
function clauseHasLiteral(clause: Clause, literal: Literal): boolean {
  const key = literalKey(literal);
  for (const lit of clause) {
    if (literalKey(lit) === key) return true;
  }
  return false;
}

The function `selectLiteral(Clauses, Forbidden)`
returns a literal from a clause from the set `Clauses` such that the variable of this literal does not occur in the set `Forbidden`.  It uses the *Jereslow-Wang heuristic* to choose the literal. The Jereslow-Wang heuristic $\texttt{JW}(l)$ of a literal $l$ in a set of clauses is defined as follows:
$$ 
\texttt{JW}(\textrm{Clauses}, l) = \sum\limits_{\{\,C \in \texttt{Clauses}\;\mid\; l \in C\;\}} \frac{1}{\;2^{|C|}\;} 
$$ 
Here, $|C|$ denotes the number of literals in the clause $C$.  The idea is to choose a literal that occurs in many clauses and therefore subsumes a lot of clauses.

In [ ]:
function selectLiteral(
  Clauses: Set<Clause>,
  Variables: Set<Variable>,
  UsedVars: Set<Variable>
): Literal | null {
  const Scores: Map<Literal, number> = new Map();

  for (const varr of Variables) {
    if (!UsedVars.has(varr)) {
      const cmp: Literal = ['¬', varr];
      Scores.set(varr, 0.0);
      Scores.set(cmp, 0.0);
      for (const C of Clauses) {
        if (clauseHasLiteral(C, cmp)) {
          Scores.set(cmp, (Scores.get(cmp) ?? 0) + Math.pow(2, -C.size));
        }
        if (clauseHasLiteral(C, varr)) {
          Scores.set(varr, (Scores.get(varr) ?? 0) + Math.pow(2, -C.size));
        }
      }
    }
  }

  // Select the literal with the maximum score
  let maxLiteral: Literal | null = null;
  let maxScore = -Infinity;
  for (const [lit, score] of Scores.entries()) {
    if (score > maxScore) {
      maxScore = score;
      maxLiteral = lit;
    }
  }

  return maxLiteral;
}

In [ ]:
const D: { [key: string]: number } = { 'a': 3, 'b': 5, 'c': 1 };
const maxKey = Object.keys(D).reduce((a, b) => (D[a] > D[b] ? a : b));
console.log(maxKey);

Given a set of clauses `Clauses` and a literal `l`, the procedure `reduce(Clauses, l)` performs all unit cuts and all unit subsumptions on clauses of of the set `Clauses` that are possible using the unit clause $\{\mathtt{l}\}$.  The resulting set of clauses is returned.  Mathematically, the function `reduce` is defined as follows:
$$\texttt{reduce}(\texttt{Clauses},l)  := 
 \Bigl\{\, C \backslash \bigl\{\overline{\,l\,}\bigr\} \;|\; C \in \texttt{Clauses} \wedge \overline{\,l\,} \in C \,\Bigr\} 
       \,\cup\, \Bigl\{\, C \in \texttt{Clauses} \mid \overline{\,l\,} \not\in C \wedge l \not\in C \Bigr\} \cup \bigl\{\{l\}\bigr\}.
$$
This function should only be called if the unit clause $\{l\}$ is an element of the set `Clauses`.

In [ ]:
function reduce(Clauses: Set<Clause>, l: Literal): Set<Clause> {
  const lBar = complement(l);
  const part1 = new Set<Clause>(
    [...Clauses]
      .filter(C => clauseHasLiteral(C, lBar))
      .map(C => normalizeClause(new Set([...C].filter(lit => literalKey(lit) !== literalKey(lBar)))))
  );
  const part2 = new Set<Clause>(
    [...Clauses].filter(C => !clauseHasLiteral(C, lBar) && !clauseHasLiteral(C, l))
  );
  const part3 = new Set<Clause>([new Set([l])]);
  return new Set([...part1, ...part2, ...part3]);
}

`Clauses` is a set of clauses.  The call `saturate(Clauses)` computes the set of those clauses that can be derived from `Clauses` via repeated applications of unit cuts or unit subsumptions.

In [ ]:
function saturate(Clauses: Set<Clause>): Set<Clause> {
  let S = new Set([...Clauses].map(normalizeClause));
  let Units = new Set([...S].filter(c => c.size === 1));
  let Used = new Set<Clause>();
  while (Units.size > 0) {
    const unit = Units.values().next().value;
    Units.delete(unit);
    Used.add(unit);
    const l = arb(unit);
    if (l === null) break;
    S = reduce(S, l as Literal);
    S = new Set([...S].map(normalizeClause));
    Units = new Set(
      [...S].filter(
        c => c.size === 1 && !( [...Used].some(u => equalClauses(u, c)) )
      )
    );
  }
  return S;
}

The function `solve(Clauses)` takes a set of clauses  as input.  The function tries to compute a variable assignment that satisfies all clauses in `Clauses`.  If this is successful, a set of unit clauses is returned.  This set of unit clauses does not contain  any complementary literals and therefore corresponds to a variable assignment satisfying all clauses.  If the set `Clauses` is unsatisfiable, then the set `{{}}` is returned instead.

The real work is done by the function `solve_recursive`.  This function takes two additional arguments:
* `Variables` is the set of all variables occurring in `Clauses`.
* `UsedVars`  is the set of those variables that have already been used in case distinctions.

In [ ]:
function solveRecursive(
  Clauses: Set<Clause>,
  Variables: Set<Variable>,
  UsedVars: Set<Variable>
): Set<Clause> {
  const S = saturate(Clauses);
  const Empty = new Set<Literal>(); // empty clause
  const Falsum = new Set<Clause>([Empty]);

  // Check if empty clause is in S (unsatisfiable)
  if ([...S].some(c => c.size === 0)) {
    return Falsum;
  }

  // Check if all clauses are unit clauses (trivial)
  if ([...S].every(c => c.size === 1)) {
    return S;
  }

  // Select literal with Jeroslow-Wang heuristic
  const l = selectLiteral(S, Variables, UsedVars);
  const lBar = complement(l);
  const p = extractVariable(l);
  const newUsed = new Set([...UsedVars, p]);

  // Recursive calls with literal true or false
  const Result = solveRecursive(new Set([...S, new Set([l])]), Variables, newUsed);
  if (!equalClauseSets(Result, Falsum)) {
    return Result;
  }
  return solveRecursive(new Set([...S, new Set([lBar])]), Variables, newUsed);
}
function equalClauseSets(a: Set<Clause>, b: Set<Clause>): boolean {
  if (a.size !== b.size) return false;
  for (const c of a) {
    if (![...b].some(x => equalClauses(x, c))) {
      return false;
    }
  }
  return true;
}

In [ ]:
function solve(Clauses: Set<Clause>): Set<Clause> {
  const Variables = new Set<Variable>();
  for (const C of Clauses) {
    for (const l of C) {
      const v = extractVariable(l);
      if (v !== null) {
        Variables.add(v);
      }
    }
  }
  return solveRecursive(Clauses, Variables, new Set());
}

The function $\texttt{toString}(S)$ takes a set $S$ as input.  The set $S$ is a set of frozensets and the function converts $S$ into a string that looks like a set of sets.  This is only used for pretty printing.

In [ ]:
function literalToStr(C: Clause): string {
  const l = arb(C);
  if (Array.isArray(l) && l[0] === '¬') {
    return `${String(l[1])} ↦ False`;
  } else {
    return `${String(l)} ↦ True`;
  }
}
function toString(S: Set<Clause>, Simplified: Set<Clause>): string | null {
  if (Simplified.size === 1) {
    const Empty = arb(Simplified);
    if (Empty && Empty.size === 0) {
      return `${prettify(S)} is unsolvable`;
    }
  } else {
    const parts = [...Simplified].map(C => literalToStr(C));
    return `{ ${parts.join(', ')} }`;
  }
  return null;
}
function prettify(Clauses: Set<Clause>): string {
  const parts = [...Clauses].map(C => {
    const literals = [...C].map(lit => String(lit));
    return `{${literals.join(', ')}}`;
  });
  return parts.join(', ');
}

In [ ]:
const c1: Clause = new Set(['r', 'p', 's']);
const c2: Clause = new Set(['r', 's']);
const c3: Clause = new Set(['p', 'q', 's']);
const c4: Clause = new Set([['¬', 'p'], ['¬', 'q']]);
const c5: Clause = new Set([['¬', 'p'], 's', ['¬', 'r']]);
const c6: Clause = new Set(['p', ['¬', 'q'], 'r']);
const c7: Clause = new Set([['¬', 'r'], ['¬', 's'], 'q']);
const c8: Clause = new Set([['¬', 'p'], ['¬', 's']]);
const c9: Clause = new Set(['p', ['¬', 'r'], ['¬', 'q']]);
const c0: Clause = new Set([['¬', 'p'], 'r', 'q', ['¬', 's']]);
const S: Set<Clause> = new Set([c0, c1, c2, c3, c4, c5, c6, c7, c8, c9]);

console.log(toString(S, solve(S)));

In [ ]:
const c11: Clause = new Set(['p', 'r', 'q', ['¬', 's']]);
const S: Set<Clause> = new Set([c0, c1, c2, c3, c4, c5, c6, c7, c8, c9, c11]);
console.log(toString(S, solve(S)));

In [ ]:
const c1: Clause = new Set(['r', 'p', 's']);
const c2: Clause = new Set(['r', 's']);
const c3: Clause = new Set(['q', 'p', 's']);
const c4: Clause = new Set([['¬', 'p'], ['¬', 'q']]);
const c5: Clause = new Set([['¬', 'p'], 's', ['¬', 'r']]);
const c6: Clause = new Set(['p', ['¬', 'q'], 'r']);
const c7: Clause = new Set([['¬', 'r'], ['¬', 's'], 'q']);
const c8: Clause = new Set(['p', 'q', 'r', 's']);
const c9: Clause = new Set(['r', ['¬', 's'], 'q']);
const c10: Clause = new Set(['s', ['¬', 'r'], ['¬', 'q']]);
const c11: Clause = new Set(['s', ['¬', 'r']]);
const c12: Clause = new Set(['r', ['¬', 's']]);

const S: Set<Clause> = new Set([c1, c2, c3, c4, c5, c6, c7, c8, c9, c10, c11, c12]);

solve(S);